[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/39_ml_systems/01_ml_system_design.ipynb)

# Bài 01 — Thiết kế hệ thống ML

**🎯 Mục tiêu**
- Mô hình chỉ là một phần nhỏ của hệ thống ML: dữ liệu, đặc trưng, huấn luyện, phục vụ, giám sát, vòng phản hồi.
- Khung thiết kế: bài toán kinh doanh → metric → dữ liệu và nhãn → baseline → ràng buộc (độ trễ, chi phí) → triển khai → giám sát.
- Hai lỗi kinh điển chỉ lộ ra khi triển khai: **rò rỉ thời điểm** (point-in-time leakage) và **lệch train/serving** (training–serving skew) — tái hiện bằng code.
- Chọn mô hình theo **ràng buộc hệ thống**: độ trễ, kích thước, độ chính xác.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/39_ml_systems

In [ ]:
import pickle
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.datasets import make_customer_churn

## 1. Mô hình chỉ là một phần nhỏ

Sculley et al. (2015, *Hidden Technical Debt in Machine Learning Systems*): trong hệ thống ML thật, code mô hình là một ô nhỏ giữa: thu thập & kiểm tra dữ liệu, tạo đặc trưng, quản lý cấu hình, hạ tầng phục vụ, giám sát, công cụ phân tích, quản lý tài nguyên...

```
 nguồn dữ liệu ─► kiểm tra & làm sạch ─► đặc trưng (feature store) ─► huấn luyện ─► đánh giá ─► đăng ký mô hình
                                              │                                                   │
                                              └──────────► phục vụ (API / batch) ◄────────────────┘
                                                               │
                                       người dùng / quyết định ─► log ─► giám sát (trôi dữ liệu, hiệu năng) ─► huấn luyện lại
```

### Khung thiết kế (dùng cho mọi dự án và capstone — chương 41)

| Bước | Câu hỏi |
|---|---|
| 1. Bài toán | Quyết định nào sẽ thay đổi nhờ mô hình? Ai dùng? Không có mô hình thì làm thế nào (baseline nghiệp vụ)? |
| 2. Metric | Metric kinh doanh (lợi nhuận, thời gian tiết kiệm) ↔ metric mô hình (AUC, MAE) ↔ metric vận hành (độ trễ, chi phí) |
| 3. Dữ liệu & nhãn | Nhãn định nghĩa thế nào, có trễ bao lâu? Đặc trưng nào **có sẵn tại thời điểm dự đoán**? |
| 4. Baseline | Luật nghiệp vụ, mô hình đơn giản — mô hình phức tạp phải vượt được |
| 5. Phục vụ | Batch (dự đoán định kỳ) hay online (theo yêu cầu)? Độ trễ, thông lượng, chi phí |
| 6. Đánh giá | Offline (tách theo thời gian) → shadow → A/B test (chương 40) |
| 7. Giám sát | Dữ liệu trôi, hiệu năng, công bằng; khi nào huấn luyện lại; ai chịu trách nhiệm |

## 2. Rò rỉ thời điểm (point-in-time leakage)

Bài toán: mỗi tháng, chọn khách hàng có nguy cơ rời bỏ trong **30 ngày tới** để gọi giữ chân. Dữ liệu: nhật ký hoạt động hằng ngày của khách hàng. Tại **ngày chụp** (snapshot) 120, đặc trưng chỉ được dùng dữ liệu **trước** ngày 120; nhãn = có rời bỏ trong ngày 121–150 không.

Một lỗi rất hay gặp khi tạo dữ liệu huấn luyện từ kho dữ liệu: tính đặc trưng trên **toàn bộ lịch sử hiện có** (đến hôm nay — ngày 180), thay vì "như tại ngày 120".

In [ ]:
def simulate_activity(n_customers=4000, n_days=180, seed=0):
    """Nhật ký hoạt động: activity[i, d] = 1 nếu khách i hoạt động ngày d; churn_day[i] (np.inf nếu không rời bỏ)."""
    rng = np.random.default_rng(seed)
    engagement = rng.beta(2, 3, n_customers)                           # mức hoạt động nền
    churn_day = np.where(rng.random(n_customers) < 0.35, rng.uniform(60, 180, n_customers), np.inf)
    days = np.arange(n_days)
    fading = np.clip((churn_day[:, None] - days[None, :]) / 60, 0, 1)  # hoạt động giảm dần 60 ngày trước khi rời bỏ
    p_active = engagement[:, None] * np.where(np.isinf(churn_day)[:, None], 1.0, fading)
    activity = (rng.random((n_customers, n_days)) < p_active).astype(int)
    return activity, churn_day


activity, churn_day = simulate_activity()
SNAPSHOT, HORIZON = 120, 30
alive = churn_day > SNAPSHOT                                           # chỉ dự đoán cho khách còn hoạt động tại ngày chụp
label = (churn_day[alive] <= SNAPSHOT + HORIZON).astype(int)
print(f"{alive.sum():,} khách còn hoạt động tại ngày {SNAPSHOT} | tỉ lệ rời bỏ trong {HORIZON} ngày tới = {label.mean():.1%}")


def features(activity, as_of):
    """Đặc trưng tính từ dữ liệu TRƯỚC ngày as_of."""
    hist = activity[:, :as_of]
    last_active = np.array([np.max(np.flatnonzero(row)) if row.any() else -1 for row in hist])
    return pd.DataFrame({"active_last_7": hist[:, -7:].sum(1), "active_last_30": hist[:, -30:].sum(1),
                         "active_total": hist.sum(1), "days_since_last": as_of - 1 - last_active,
                         "trend": hist[:, -30:].mean(1) - hist[:, -60:-30].mean(1)})


X_correct = features(activity, SNAPSHOT)[alive]                        # đúng: như tại ngày 120
X_leaky = features(activity, 180)[alive]                               # SAI: dùng dữ liệu tới hôm nay (ngày 180)
idx_tr, idx_te = train_test_split(np.arange(alive.sum()), test_size=0.3, random_state=0, stratify=label)
results = {}
for name, Xf in [("đặc trưng đúng thời điểm", X_correct), ("đặc trưng rò rỉ (tính tới ngày 180)", X_leaky)]:
    model = HistGradientBoostingClassifier(random_state=0).fit(Xf.iloc[idx_tr], label[idx_tr])
    offline = roc_auc_score(label[idx_te], model.predict_proba(Xf.iloc[idx_te])[:, 1])
    production = roc_auc_score(label[idx_te], model.predict_proba(X_correct.iloc[idx_te])[:, 1])   # khi triển khai chỉ có dữ liệu tới ngày 120
    results[name] = (offline, production)
    print(f"{name:36s}: AUC offline = {offline:.3f} | AUC khi triển khai = {production:.3f}")

Mô hình rò rỉ có AUC offline "đẹp như mơ" (nó nhìn thấy khách hàng **đã ngừng hoạt động** sau ngày 120), nhưng khi triển khai — chỉ có dữ liệu tới hôm nay — hiệu năng sụp mạnh, thậm chí tệ hơn mô hình đúng. Lỗi này **không** lộ ra bằng train/test split ngẫu nhiên; chỉ lộ ra khi xây dữ liệu **theo đúng thời điểm** ("point-in-time correct" — tính năng cốt lõi của feature store) và đánh giá theo **thời gian**: train trên các ngày chụp cũ, test trên ngày chụp mới hơn.

## 3. Lệch train/serving (training–serving skew)

Cùng mô hình, nhưng đường ống tính đặc trưng ở môi trường phục vụ khác lúc huấn luyện. Ví dụ: lúc train, giá trị thiếu của `monthly_charges` được điền **trung vị** trong pipeline; đội backend viết lại logic ở dịch vụ khác và điền **0**.

In [ ]:
raw = make_customer_churn(20000, seed=0)
X = pd.DataFrame({"tenure": raw.tenure_months, "monthly_charges": raw.monthly_charges, "support_calls": raw.support_calls,
                  "month_to_month": (raw.contract == "month-to-month").astype(int)}).astype(float)
y = raw.churn.values
rng = np.random.default_rng(1)
X.loc[rng.random(len(X)) < 0.15, "monthly_charges"] = np.nan             # 15% thiếu (vd khách mới chưa có hoá đơn)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)
same_pipeline = pipe.predict_proba(X_te)[:, 1]
serving_rewrite = pipe.predict_proba(X_te.fillna({"monthly_charges": 0.0}))[:, 1]   # logic phục vụ viết lại: điền 0
missing = X_te.monthly_charges.isna().values
print(f"AUC — cùng pipeline: {roc_auc_score(y_te, same_pipeline):.4f} | logic phục vụ viết lại: {roc_auc_score(y_te, serving_rewrite):.4f}")
print(f"xác suất TB của khách thiếu dữ liệu: cùng pipeline = {same_pipeline[missing].mean():.3f} | phục vụ viết lại = {serving_rewrite[missing].mean():.3f}")

AUC tổng thể giảm không nhiều (lỗi chỉ ảnh hưởng 15% khách) — đủ để không ai để ý trên dashboard — nhưng xác suất của nhóm khách thiếu dữ liệu bị **lệch có hệ thống** (cước 0 → mô hình nghĩ khách trả rất ít → rủi ro thấp hơn thật). Nhóm này có thể là khách **mới** — đúng nhóm quan trọng. Phòng tránh:
- **Đóng gói tiền xử lý cùng mô hình** (Pipeline — bài 03) — phục vụ gọi đúng đối tượng đó, không viết lại logic.
- Dùng chung một thư viện/feature store cho train và serve; **log đặc trưng lúc phục vụ** và so phân phối với lúc train (chương 40).
- Kiểm thử: với cùng bản ghi đầu vào, dịch vụ phục vụ phải cho đúng dự đoán như notebook huấn luyện.

## 4. Chọn mô hình theo ràng buộc hệ thống

Yêu cầu giả định: API chấm điểm từng khách khi họ mở ứng dụng, **độ trễ p99 < 5 ms** trên một CPU, mô hình < 50 MB.

In [ ]:
candidates = {
    "Logistic regression": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000)),
    "Gradient boosting (100 cây)": make_pipeline(SimpleImputer(strategy="median"), HistGradientBoostingClassifier(random_state=0)),
    "Random forest (500 cây sâu)": make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(500, random_state=0, n_jobs=1)),
}
rows = []
single = X_te.iloc[[0]]
for name, model in candidates.items():
    model.fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
    latencies = []
    for _ in range(200):
        t0 = time.perf_counter()
        model.predict_proba(single)
        latencies.append((time.perf_counter() - t0) * 1000)
    t0 = time.perf_counter()
    model.predict_proba(X_te)
    batch_ms = (time.perf_counter() - t0) * 1000
    rows.append({"mô hình": name, "ROC-AUC": auc, "độ trễ p50 (ms)": np.percentile(latencies, 50), "độ trễ p99 (ms)": np.percentile(latencies, 99),
                 f"batch {len(X_te):,} dòng (ms)": batch_ms, "kích thước (MB)": len(pickle.dumps(model)) / 2**20})
print(pd.DataFrame(rows).round(3).to_string(index=False))

Trên dữ liệu này, mô hình đơn giản **vừa chính xác nhất** vừa nhanh và nhỏ hơn rất nhiều; rừng ngẫu nhiên 500 cây sâu nặng hàng trăm MB, độ trễ p99 vượt ngân sách 5 ms — và còn kém chính xác hơn (cây sâu overfit). Nếu chỉ nhìn AUC trên notebook, ta sẽ không thấy những chi phí này. Lưu ý: độ trễ đo **trong** notebook chưa tính chi phí mạng, tuần tự hoá, tải đặc trưng — ngân sách thật chặt hơn. Một mô hình chỉ tốt hơn 0.001 AUC nhưng chậm gấp 100 lần hiếm khi đáng.

### Batch inference vs online inference

Hệ thống ML có hai đường ống: **pipeline huấn luyện** (dữ liệu lịch sử → đặc trưng → mô hình → đăng ký) và **pipeline suy luận** (dữ liệu mới → *cùng* đặc trưng → mô hình → quyết định). Suy luận chạy theo hai kiểu: **batch** — một lần gọi cho cả bảng (vd mỗi đêm chấm điểm mọi khách hàng, ghi vào bảng kết quả) — hoặc **online** — một lần gọi cho mỗi yêu cầu. Cùng một mô hình, chi phí **cho mỗi dự đoán** khác hẳn nhau:

In [ ]:
import matplotlib.pyplot as plt

from ml_course.visualization import set_style

set_style()
table = pd.DataFrame(rows).set_index("mô hình")
short = [name.split(" (")[0] for name in table.index]
per_pred_online = table["độ trễ p50 (ms)"] * 1000                     # µs cho một dự đoán khi gọi từng dòng
per_pred_batch = table[f"batch {len(X_te):,} dòng (ms)"] / len(X_te) * 1000
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
x = np.arange(len(table))
axes[0].bar(x - 0.2, per_pred_online, 0.4, label="online: từng dòng (p50)")
axes[0].bar(x + 0.2, per_pred_batch, 0.4, label="batch: cả bảng ÷ số dòng")
axes[0].set_yscale("log")
axes[0].set_xticks(x, short)
axes[0].set_ylabel("µs cho mỗi dự đoán (thang log)")
axes[0].set_title("Chi phí mỗi dự đoán")
axes[0].legend()
axes[1].scatter(table["độ trễ p99 (ms)"], table["ROC-AUC"], s=80)
for name, (lat, auc) in zip(short, table[["độ trễ p99 (ms)", "ROC-AUC"]].to_numpy()):
    axes[1].annotate(name, (lat, auc), textcoords="offset points", xytext=(6, 4))
axes[1].axvline(5, color="C3", ls="--", label="ngân sách p99 = 5 ms")
axes[1].set_xscale("log")
axes[1].set_xlabel("độ trễ p99 của một dự đoán online (ms, thang log)")
axes[1].set_ylabel("ROC-AUC")
axes[1].set_title("Đánh đổi chính xác – độ trễ")
axes[1].legend()
plt.tight_layout()
plt.show()
print("batch rẻ hơn online (mỗi dự đoán):", ", ".join(f"{n} ×{a / b:,.0f}" for n, a, b in zip(short, per_pred_online, per_pred_batch)))

Gọi từng dòng phải trả **chi phí cố định** của mỗi lần gọi (kiểm tra đầu vào, tạo bảng, duyệt cây…) cho chỉ một dự đoán; chạy cả bảng thì chi phí đó được chia cho hàng nghìn dòng và tận dụng được vector hoá. Vì vậy: nếu quyết định không cần kết quả tức thì (danh sách khách cần gọi giữ chân mỗi sáng), hãy **chấm điểm theo lô** và lưu kết quả; chỉ dùng online khi quyết định phụ thuộc vào dữ liệu vừa xảy ra (gợi ý khi mở ứng dụng, chặn giao dịch).


Các lựa chọn kiến trúc phục vụ:

| | Batch | Online (API) | Streaming |
|---|---|---|---|
| Khi nào | dự đoán định kỳ (hằng đêm), không cần tức thì | cần kết quả theo yêu cầu | sự kiện liên tục (gian lận giao dịch) |
| Độ trễ | phút–giờ | ms–s | ms |
| Đặc trưng | tính lại hàng loạt | phải có sẵn tức thì (feature store online) | tổng hợp theo cửa sổ thời gian |
| Ví dụ | danh sách khách cần gọi giữ chân | gợi ý khi mở app | chặn giao dịch gian lận |

## 🧠 Kiểm tra nhanh

1. Kể các thành phần của một hệ thống ML ngoài mô hình.
2. Rò rỉ thời điểm là gì? Vì sao train/test ngẫu nhiên không phát hiện được?
3. Lệch train/serving xảy ra thế nào? Ba cách phòng tránh?
4. Vì sao phải đo độ trễ và kích thước mô hình, không chỉ AUC?
5. Khi nào chọn batch, online, streaming?

## 📌 Tóm tắt

- Hệ thống ML = dữ liệu + đặc trưng + huấn luyện + phục vụ + giám sát + phản hồi; mô hình là phần nhỏ.
- Đặc trưng phải tính "như tại thời điểm dự đoán"; đánh giá theo thời gian; tiền xử lý đóng gói cùng mô hình để tránh lệch train/serving.
- Chọn mô hình theo cả metric kinh doanh và ràng buộc vận hành (độ trễ, kích thước, chi phí).

## 📚 Tài liệu tham khảo

- Sculley et al. (2015) — *Hidden Technical Debt in Machine Learning Systems*
- Huyen (2022) — *Designing Machine Learning Systems* (O'Reilly)
- Google — *Rules of Machine Learning: Best Practices for ML Engineering* (Zinkevich)
- Breck et al. (2017) — *The ML Test Score: A Rubric for ML Production Readiness and Technical Debt Reduction*
- Kleppmann (2017) — *Designing Data-Intensive Applications* (nền tảng hệ thống dữ liệu)